In [2]:
import pandas as pd
from IPython.display import display

# Если данные ещё не в памяти:
edges = pd.read_parquet("edges.parquet")
nodes = pd.read_parquet("nodes.parquet")
transactions = pd.read_parquet("transactions.parquet")

dfs = {"edges": edges, "nodes": nodes, "transactions": transactions}

# показывать все строки и колонки без обрезки
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def summary(df):
    return pd.DataFrame({
        "dtype":     df.dtypes.astype(str),
        "non_null":  df.notna().sum(),
        "nulls":     df.isna().sum(),
        "null_%":    (df.isna().mean() * 100).round(2),
        "n_unique":  df.nunique(dropna=False),
        "example":   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

for name, df in dfs.items():
    print(f"\n{'='*80}\n{name}: {df.shape[0]:,} строк × {df.shape[1]} колонок\n{'='*80}")
    display(summary(df))
    print("Дубликатов строк:", df.duplicated().sum())


edges: 3,119 строк × 5 колонок


,dtype,non_null,nulls,null_%,n_unique,example
src,int64,3119,0,0.00,694,"100,000,003,684,369,104.00"
dst,int64,3119,0,0.00,2206,"100,000,003,037,660,096.00"
sum_kzt,float64,3119,0,0.00,1484,"53,000.00"
n_tx,int64,3119,0,0.00,22,1.00
depth,int8,3119,0,0.00,4,1.00


Дубликатов строк: 0

nodes: 2,248 строк × 3 колонок


,dtype,non_null,nulls,null_%,n_unique,example
gid,int64,2248,0,0.00,2248,100000000343175100
depth,int64,2248,0,0.00,5,0
is_seed,bool,2248,0,0.00,2,True


Дубликатов строк: 0

transactions: 4,840 строк × 4 колонок


,dtype,non_null,nulls,null_%,n_unique,example
src,int64,4840,0,0.00,694,100000002175422100
dst,int64,4840,0,0.00,2206,100000003004487100
date,object,4840,0,0.00,31,2026-07-01
sum_kzt,float64,4840,0,0.00,1583,"50,000.00"


Дубликатов строк: 97


In [3]:
import numpy as np, pandas as pd, networkx as nx
from IPython.display import display, IFrame

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

nodes_ = nodes.copy()
e = edges.copy()
tx = transactions.copy()
tx["date"] = pd.to_datetime(tx["date"])
nodes_["is_seed"] = nodes_["is_seed"].astype(bool)

seeds = set(nodes_.loc[nodes_.is_seed, "gid"])
e["from_seed"] = e["src"].isin(seeds)

print("Дубли пар src→dst в edges:", e.duplicated(["src", "dst"]).sum())
print("Петли src==dst:", (e.src == e.dst).sum())

Дубли пар src→dst в edges: 0
Петли src==dst: 0


In [4]:
out_ = e.groupby("src").agg(out_sum=("sum_kzt", "sum"), out_tx=("n_tx", "sum"), out_deg=("dst", "nunique"))
in_  = e.groupby("dst").agg(in_sum=("sum_kzt", "sum"),  in_tx=("n_tx", "sum"),  in_deg=("src", "nunique"))
in_seed = (e[e.from_seed].groupby("dst")
           .agg(in_from_seed_sum=("sum_kzt", "sum"), in_from_seed_deg=("src", "nunique")))

# активность по датам (как отправитель или получатель)
act = pd.concat([tx[["src", "date"]].rename(columns={"src": "gid"}),
                 tx[["dst", "date"]].rename(columns={"dst": "gid"})])
act = act.groupby("gid")["date"].agg(first_date="min", last_date="max", active_days="nunique")

reg = nodes_.set_index("gid").join([out_, in_, in_seed, act])
num_cols = ["out_sum", "out_tx", "out_deg", "in_sum", "in_tx", "in_deg", "in_from_seed_sum", "in_from_seed_deg"]
reg[num_cols] = reg[num_cols].fillna(0)

# уникальные контрагенты (и входящие, и исходящие)
pairs = pd.concat([e[["src", "dst"]].rename(columns={"src": "gid", "dst": "cp"}),
                   e[["dst", "src"]].rename(columns={"dst": "gid", "src": "cp"})])
reg["n_counterparties"] = pairs.groupby("gid")["cp"].nunique()
reg["n_counterparties"] = reg["n_counterparties"].fillna(0).astype(int)

# производные метрики
reg["total_volume"] = reg.in_sum + reg.out_sum            # оборот: вход + выход
reg["throughput"]   = np.minimum(reg.in_sum, reg.out_sum) # сколько «прошло насквозь»
reg["net_flow"]     = reg.in_sum - reg.out_sum            # >0 осело, <0 отдал больше, чем видно на входе
reg["pass_ratio"]   = np.where(reg.in_sum > 0, reg.out_sum / reg.in_sum, np.nan)

# флаги артефактов данных
reg["flag_truncated"]   = (reg.depth == 4) & (reg.out_deg == 0)   # обрыв обхода, а не «сток»
reg["flag_seed_no_out"] = reg.is_seed & (reg.out_deg == 0)
reg["flag_out_gt_in"]   = (reg.out_sum > reg.in_sum) & ~reg.is_seed
reg["flag_isolated"]    = (reg.in_deg + reg.out_deg) == 0

print(f"Узлов: {len(reg):,}")
print("Обрыв 4-го колена:", reg.flag_truncated.sum(), "(в ТЗ 444)")
print("Seed без исходящих:", reg.flag_seed_no_out.sum(), "(в ТЗ 31)")
print("Отдают больше, чем получили (не seed):", reg.flag_out_gt_in.sum())
print("Изолированные узлы:", reg.flag_isolated.sum())
display(reg.sort_values("total_volume", ascending=False).head(10))

Узлов: 2,248
Обрыв 4-го колена: 444 (в ТЗ 444)
Seed без исходящих: 31 (в ТЗ 31)
Отдают больше, чем получили (не seed): 335
Изолированные узлы: 19


,depth,is_seed,out_sum,out_tx,out_deg,in_sum,in_tx,in_deg,in_from_seed_sum,in_from_seed_deg,first_date,last_date,active_days,n_counterparties,total_volume,throughput,net_flow,pass_ratio,flag_truncated,flag_seed_no_out,flag_out_gt_in,flag_isolated
gid,,,,,,,,,,,,,,,,,,,,,,
100000000331309100,2,False,"23,001,375",126,99,"984,635",8,5,0,0,2026-07-01,2026-07-31,29,104,"23,986,010","984,635","-22,016,740",23,False,False,True,False
100000002224132100,2,False,"12,115,000",25,4,"3,951,020",10,5,0,0,2026-07-02,2026-07-29,15,6,"16,066,020","3,951,020","-8,163,980",3,False,False,True,False
100000003684369100,0,True,"8,588,655",67,62,"3,848,436",58,24,"233,000",1,2026-07-12,2026-07-28,16,85,"12,437,091","3,848,436","-4,740,219",2,False,False,False,False
100000003016635100,0,True,"9,414,081",151,73,"586,981",36,8,0,0,2026-07-03,2026-07-23,21,75,"10,001,062","586,981","-8,827,100",16,False,False,False,False
100000005242320100,2,False,"8,936,295",29,21,"365,700",2,2,0,0,2026-07-01,2026-07-21,16,23,"9,301,995","365,700","-8,570,595",24,False,False,True,False
100000000437046100,2,False,"7,606,224",73,42,"1,107,200",20,9,0,0,2026-07-10,2026-07-31,21,48,"8,713,424","1,107,200","-6,499,024",7,False,False,True,False
100000008603629100,2,False,"4,986,156",74,61,"1,817,300",45,19,0,0,2026-07-03,2026-07-31,27,73,"6,803,456","1,817,300","-3,168,856",3,False,False,True,False
100000004400305100,2,False,"5,830,181",101,82,"222,800",9,7,0,0,2026-07-03,2026-07-29,22,87,"6,052,981","222,800","-5,607,381",26,False,False,True,False
100000002963189100,3,False,"5,219,000",15,7,"490,000",3,1,0,0,2026-07-08,2026-07-25,7,7,"5,709,000","490,000","-4,729,000",11,False,False,True,False


In [5]:
tx_agg = tx.groupby(["src", "dst"]).agg(
    tx_sum=("sum_kzt", "sum"), tx_n=("sum_kzt", "size"),
    min_tx=("sum_kzt", "min"), max_tx=("sum_kzt", "max"),
    first_date=("date", "min"), last_date=("date", "max"), active_days=("date", "nunique"))

edge_reg = (e.merge(tx_agg, on=["src", "dst"], how="left")
             .merge(reg[["depth", "is_seed"]].add_prefix("src_"), left_on="src", right_index=True)
             .merge(reg[["depth", "is_seed"]].add_prefix("dst_"), left_on="dst", right_index=True))

pair_set = set(zip(e.src, e.dst))
edge_reg["mutual"] = [(d, s) in pair_set for s, d in zip(edge_reg.src, edge_reg.dst)]  # A→B и B→A
edge_reg["back_edge"] = edge_reg.dst_depth <= edge_reg.src_depth                        # деньги идут «назад/вбок»
edge_reg["sum_mismatch"] = (edge_reg.sum_kzt - edge_reg.tx_sum).abs() > 1
edge_reg = edge_reg.sort_values("sum_kzt", ascending=False)

print("Расхождений edges vs transactions:", edge_reg.sum_mismatch.sum())
print("Взаимных пар:", edge_reg.mutual.sum(), "| обратных/боковых рёбер:", edge_reg.back_edge.sum())
display(edge_reg.head(20))

# матрица потоков между коленами: откуда и куда идут деньги
print("\nОборот между коленами (строки: колено отправителя, колонки: колено получателя):")
display(pd.pivot_table(edge_reg, index="src_depth", columns="dst_depth",
                       values="sum_kzt", aggfunc="sum", fill_value=0))

# граф
G = nx.from_pandas_edgelist(e, "src", "dst", edge_attr=["sum_kzt", "n_tx", "depth"], create_using=nx.DiGraph)
G.add_nodes_from(reg.index)

comps = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
reg["component"] = pd.Series({n: i for i, c in enumerate(comps) for n in c})
reg["component_size"] = reg.groupby("component")["depth"].transform("size")
print(f"\nКомпонент: {len(comps)} | размеры топ-5: {[len(c) for c in comps[:5]]}")
display(reg.groupby("component").agg(n=("depth", "size"), seeds=("is_seed", "sum"),
                                     volume=("out_sum", "sum")).head(16))

Расхождений edges vs transactions: 0
Взаимных пар: 354 | обратных/боковых рёбер: 745


,src,dst,sum_kzt,n_tx,depth,from_seed,tx_sum,tx_n,min_tx,max_tx,first_date,last_date,active_days,src_depth,src_is_seed,dst_depth,dst_is_seed,mutual,back_edge,sum_mismatch
2090,100000002224132100,100000004892144100,"4,400,000",6,3,False,"4,400,000",6,"200,000","1,000,000",2026-07-02,2026-07-16,4,2,False,3,False,True,False,False
1537,100000007594394100,100000006274203100,"3,376,312",40,3,False,"3,376,312",40,"20,000","250,000",2026-07-11,2026-07-31,19,2,False,3,False,True,False,False
1132,100000007442518100,100000008258083100,"3,000,000",1,2,False,"3,000,000",1,"3,000,000","3,000,000",2026-07-21,2026-07-21,1,1,False,2,False,False,False,False
561,100000003835149100,100000002224132100,"2,900,720",6,2,False,"2,900,720",6,"136,450","1,155,800",2026-07-08,2026-07-29,6,1,False,2,False,False,False,False
862,100000000274067100,100000002958343100,"2,900,000",4,2,False,"2,900,000",4,"100,000","2,000,000",2026-07-01,2026-07-13,4,1,False,2,False,True,False,False
2103,100000002224132100,100000000594783100,"2,635,000",10,3,False,"2,635,000",10,"5,000","1,000,000",2026-07-02,2026-07-27,7,2,False,3,False,True,False,False
1330,100000002224132100,100000004603109100,"2,580,000",6,3,False,"2,580,000",6,"180,000","1,000,000",2026-07-02,2026-07-08,5,2,False,3,False,True,False,False
2034,100000002224132100,100000000552584100,"2,500,000",3,3,False,"2,500,000",3,"500,000","1,000,000",2026-07-08,2026-07-16,2,2,False,3,False,False,False,False
186,100000008637537100,100000001732159100,"2,283,100",28,1,True,"2,283,100",28,"10,000","318,000",2026-07-01,2026-07-17,10,0,True,0,True,True,True,False
2441,100000004722765100,100000005075949100,"2,226,000",7,4,False,"2,226,000",7,"10,000","984,000",2026-07-09,2026-07-23,4,3,False,4,False,False,False,False



Оборот между коленами (строки: колено отправителя, колонки: колено получателя):


dst_depth,0,1,2,3,4
src_depth,,,,,
0,"6,609,582","48,684,596",0,0,0
1,"5,506,206","6,214,120","54,064,002",0,0
2,"1,318,644","24,926,442","10,556,837","122,189,653",0
3,"1,414,735","3,031,847","16,644,483","8,056,700","56,672,165"



Компонент: 35 | размеры топ-5: [1877, 270, 17, 13, 6]


,n,seeds,volume
component,,,
0,1877,46,"342,125,465"
1,270,1,"10,178,044"
2,17,1,"2,977,887"
3,13,1,"1,392,916"
4,6,1,"275,885"
5,6,1,"1,615,030"
6,6,1,"383,300"
7,6,1,"1,527,967"
8,5,2,"108,007"


In [6]:
top100 = reg.sort_values("total_volume", ascending=False).head(100).reset_index()
top100.insert(0, "rank", range(1, 101))
cols = ["rank", "gid", "depth", "is_seed", "total_volume", "in_sum", "out_sum", "net_flow", "pass_ratio",
        "in_deg", "out_deg", "n_counterparties", "in_from_seed_sum", "active_days",
        "component", "flag_truncated", "flag_out_gt_in"]
display(top100[cols])

graph_turnover = e.sum_kzt.sum()
print(f"Оборот графа: {graph_turnover:,.0f} KZT (в ТЗ 365 890 012)")
print(f"Доля топ-100 в сумме in+out: {top100.total_volume.sum() / (2 * graph_turnover):.1%}")

# чем топ-100 отличается от сети в целом
print("\nРаспределение по коленам: топ-100 vs вся сеть (%)")
display(pd.DataFrame({"top100_%": top100.depth.value_counts(normalize=True).sort_index() * 100,
                      "all_%":    reg.depth.value_counts(normalize=True).sort_index() * 100}))
print("В топ-100: seed", top100.is_seed.sum(), "| обрыв 4-го колена", top100.flag_truncated.sum(),
      "| out>in", top100.flag_out_gt_in.sum())

# насколько топ зависит от выбранной метрики
print("\nПересечение топ-100 по обороту с топ-100 по другим метрикам:")
top_set = set(top100.gid)
for m in ["in_sum", "out_sum", "throughput", "n_counterparties", "in_deg", "out_deg", "in_from_seed_sum"]:
    print(f"  {m:18s}: {len(top_set & set(reg.nlargest(100, m).index))}")

,rank,gid,depth,is_seed,total_volume,in_sum,out_sum,net_flow,pass_ratio,in_deg,out_deg,n_counterparties,in_from_seed_sum,active_days,component,flag_truncated,flag_out_gt_in
0,1,100000000331309100,2,False,"23,986,010","984,635","23,001,375","-22,016,740",23,5,99,104,0,29,0,False,True
1,2,100000002224132100,2,False,"16,066,020","3,951,020","12,115,000","-8,163,980",3,5,4,6,0,15,0,False,True
2,3,100000003684369100,0,True,"12,437,091","3,848,436","8,588,655","-4,740,219",2,24,62,85,"233,000",16,0,False,False
3,4,100000003016635100,0,True,"10,001,062","586,981","9,414,081","-8,827,100",16,8,73,75,0,21,0,False,False
4,5,100000005242320100,2,False,"9,301,995","365,700","8,936,295","-8,570,595",24,2,21,23,0,16,0,False,True
5,6,100000000437046100,2,False,"8,713,424","1,107,200","7,606,224","-6,499,024",7,9,42,48,0,21,0,False,True
6,7,100000008603629100,2,False,"6,803,456","1,817,300","4,986,156","-3,168,856",3,19,61,73,0,27,0,False,True
7,8,100000004400305100,2,False,"6,052,981","222,800","5,830,181","-5,607,381",26,7,82,87,0,22,0,False,True
8,9,100000002963189100,3,False,"5,709,000","490,000","5,219,000","-4,729,000",11,1,7,7,0,7,0,False,True
9,10,100000008710791100,2,False,"5,703,021","181,510","5,521,511","-5,340,001",30,3,23,26,0,10,0,False,True


Оборот графа: 365,890,012 KZT (в ТЗ 365 890 012)
Доля топ-100 в сумме in+out: 47.5%

Распределение по коленам: топ-100 vs вся сеть (%)


,top100_%,all_%
depth,,
0,11,4
1,23,21
2,34,21
3,27,35
4,5,20


В топ-100: seed 11 | обрыв 4-го колена 5 | out>in 58

Пересечение топ-100 по обороту с топ-100 по другим метрикам:
  in_sum            : 44
  out_sum           : 72
  throughput        : 54
  n_counterparties  : 47
  in_deg            : 32
  out_deg           : 49
  in_from_seed_sum  : 17


In [11]:
from pyvis.network import Network

DEPTH_COLORS = {0: "#d62728", 1: "#ff7f0e", 2: "#2ca02c", 3: "#1f77b4", 4: "#9467bd"}
fmt = lambda x: f"{x:,.0f}".replace(",", " ")
VMAX = np.log1p(reg.total_volume.max())
EMAX = np.log1p(e.sum_kzt.max())

def draw(G_sub, path, highlight=(), label_nodes=None):
    label_nodes = (set(label_nodes) if label_nodes is not None else set()) | set(highlight)
    net = Network(height="850px", width="100%", directed=True, select_menu=True, cdn_resources="in_line")
    for n in G_sub.nodes():
        r = reg.loc[n]
        title = (f"gid {n}{'  [SEED]' if r.is_seed else ''}\nколено: {int(r.depth)}\n"
                 f"получил: {fmt(r.in_sum)} KZT от {int(r.in_deg)}\n"
                 f"отправил: {fmt(r.out_sum)} KZT на {int(r.out_deg)}\n"
                 f"оборот: {fmt(r.total_volume)} KZT"
                 + ("\n⚠ обрыв 4-го колена (исходящие не выгружены)" if r.flag_truncated else ""))
        net.add_node(int(n), label=str(n) if (r.is_seed or n in label_nodes) else " ",
                     title=title, color=DEPTH_COLORS.get(int(r.depth), "#999"),
                     size=float(5 + 30 * np.log1p(r.total_volume) / VMAX),
                     shape="star" if r.is_seed else "dot",
                     borderWidth=5 if n in highlight else 1)
    for u, v, d in G_sub.edges(data=True):
        net.add_edge(int(u), int(v), width=float(0.5 + 6 * np.log1p(d["sum_kzt"]) / EMAX),
                     title=f"{u} → {v}\n{fmt(d['sum_kzt'])} KZT, {int(d['n_tx'])} тр.", color="#999")
    net.set_options("""{
      "physics": {"solver": "forceAtlas2Based",
                  "forceAtlas2Based": {"gravitationalConstant": -40, "springLength": 80},
                  "stabilization": {"iterations": 400}},
      "edges": {"smooth": false, "arrows": {"to": {"enabled": true, "scaleFactor": 0.4}}},
      "interaction": {"hover": true}
    }""")
    # pyvis на Windows пишет в cp1251 — сохраняем сами в UTF-8
    with open(path, "w", encoding="utf-8") as f:
        f.write(net.generate_html())
    return path

# 1) вся сеть: цвет = колено, звезда = seed, размер = оборот, толщина = сумма
draw(G, "graph_full.html", label_nodes=top100.gid)
display(IFrame("graph_full.html", width="100%", height=870))

In [12]:
# 2) топ-100 + все seed и связи между ними (легче читать)
sub_nodes = set(top100.gid) | seeds
draw(G.subgraph(sub_nodes), "graph_top100.html", highlight=set(top100.gid[:20]), label_nodes=sub_nodes)
display(IFrame("graph_top100.html", width="100%", height=870))

In [13]:
findings = []
def note(point, what, value, conclusion=""):
    findings.append({"пункт": point, "проверка": what, "значение": value, "вывод": conclusion})
    print(f"[{point}] {what}: {value}" + (f"\n     → {conclusion}" if conclusion else ""))

# чтобы маленькие доли не округлялись до 0
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.4g}")
TOTAL = e.sum_kzt.sum()

In [14]:
transit = reg.pass_ratio.between(0.8, 1.2) & (reg.in_sum > 0) & (reg.out_sum > 0)
reg["volume_dedup"] = np.maximum(reg.in_sum, reg.out_sum)   # сколько денег прошло через узел, без двойного счёта

top_tv = set(reg.nlargest(100, "total_volume").index)
top_dd = set(reg.nlargest(100, "volume_dedup").index)
t_top, t_all = transit[list(top_tv)].mean(), transit.mean()

note(1, "доля транзитных (pass 0.8–1.2): топ-100 по in+out vs вся сеть", f"{t_top:.0%} vs {t_all:.0%}",
     "транзит перепредставлен в топе" if t_top > 2 * t_all else "сильного перекоса нет")
note(1, "пересечение топ-100 по in+out и по max(in,out)", len(top_tv & top_dd),
     "ранжирование заметно меняется" if len(top_tv & top_dd) < 85 else "почти не меняется")

cmp = pd.DataFrame({"rank_in_out": reg.total_volume.rank(ascending=False),
                    "rank_dedup":  reg.volume_dedup.rank(ascending=False),
                    "pass_ratio":  reg.pass_ratio})
cmp["shift"] = cmp.rank_in_out - cmp.rank_dedup
print("\nСильнее всего опускаются при дедупликации:")
display(cmp.loc[list(top_tv)].sort_values("shift").head(10))

[1] доля транзитных (pass 0.8–1.2): топ-100 по in+out vs вся сеть: 2% vs 3%
     → сильного перекоса нет
[1] пересечение топ-100 по in+out и по max(in,out): 91
     → почти не меняется

Сильнее всего опускаются при дедупликации:


,rank_in_out,rank_dedup,pass_ratio,shift
gid,,,,
100000008165763100,53,111,1.088,-58
100000004135268100,58,115,0.8073,-57
100000007629096100,82,132,0.7117,-50
100000004221668100,79,123,0.6483,-44
100000002849158100,65,106,1.816,-40.5
100000001887715100,62,102,0.5633,-40
100000000791285100,89,125,0.5804,-36
100000006889963100,76,95,0.3099,-19
100000008512552100,91,109,3.302,-18


In [15]:
ng = reg[~reg.is_seed]
gap = ng[ng.flag_out_gt_in].assign(gap=lambda d: d.out_sum - d.in_sum)

note(2, "не-seed узлов с out > in", len(gap), "у них не виден часть входа, баланс считать нельзя")
note(2, "невидимый вход у них в сумме, KZT", fmt(gap.gap.sum()), f"= {gap.gap.sum() / TOTAL:.1%} оборота графа")

first_out = tx.groupby("src").date.min().reindex(gap.index)
first_in  = tx.groupby("dst").date.min().reindex(gap.index)
n_early = int((first_out < first_in).sum())
note(2, "из них первый исходящий раньше первого видимого входящего", n_early,
     "деньги были на счёте до выборки, т.е. вход из-за пределов графа")

print("\nПо коленам:"); display(gap.depth.value_counts().sort_index())
display(gap.sort_values("gap", ascending=False).head(15)
        [["depth", "in_sum", "out_sum", "gap", "in_deg", "out_deg"]])

[2] не-seed узлов с out > in: 335
     → у них не виден часть входа, баланс считать нельзя
[2] невидимый вход у них в сумме, KZT: 229 908 004
     → = 62.8% оборота графа
[2] из них первый исходящий раньше первого видимого входящего: 186
     → деньги были на счёте до выборки, т.е. вход из-за пределов графа

По коленам:


depth
1     80
2    110
3    145
Name: count, dtype: int64

,depth,in_sum,out_sum,gap,in_deg,out_deg
gid,,,,,,
100000000331309100,2,"984,635","23,001,375","22,016,740",5,99
100000005242320100,2,"365,700","8,936,295","8,570,595",2,21
100000002224132100,2,"3,951,020","12,115,000","8,163,980",5,4
100000000437046100,2,"1,107,200","7,606,224","6,499,024",9,42
100000004400305100,2,"222,800","5,830,181","5,607,381",7,82
100000008710791100,2,"181,510","5,521,511","5,340,001",3,23
100000008547948100,2,"212,665","5,185,600","4,972,935",6,26
100000002963189100,3,"490,000","5,219,000","4,729,000",1,7
100000003835149100,1,"248,500","4,600,440","4,351,940",1,2


In [16]:
no_out    = reg.out_deg == 0
trunc     = reg.flag_truncated
real_sink = no_out & (reg.depth < 4) & ~reg.is_seed     # у колен 0–3 исходящие выгружены, значит сток видимый
reg["flag_real_sink"] = real_sink

note(3, "узлов без исходящих всего", int(no_out.sum()))
note(3, "из них обрыв 4-го колена", int(trunc.sum()), "не terminal, а «нет данных»")
note(3, "кандидаты в настоящие стоки (колено<4, не seed, out=0)", int(real_sink.sum()),
     f"сумма входа {fmt(reg.loc[real_sink, 'in_sum'].sum())} KZT")
note(3, "доля денег графа, ушедших в обрезанные узлы", f"{reg.loc[trunc, 'in_sum'].sum() / TOTAL:.1%}")
note(3, "обрезанных в топ-100 по входу", int(trunc[reg.nlargest(100, 'in_sum').index].sum()),
     "наивное правило «много пришло и не ушло» поймает их")

[3] узлов без исходящих всего: 1554
[3] из них обрыв 4-го колена: 444
     → не terminal, а «нет данных»
[3] кандидаты в настоящие стоки (колено<4, не seed, out=0): 1079
     → сумма входа 136 739 480 KZT
[3] доля денег графа, ушедших в обрезанные узлы: 15.5%
[3] обрезанных в топ-100 по входу: 17
     → наивное правило «много пришло и не ушло» поймает их


In [17]:
idx = {g: i for i, g in enumerate(reg.index)}
s = e.src.map(idx).values; d = e.dst.map(idx).values; w = e.sum_kzt.values
denom = np.maximum(reg.in_sum.values, reg.out_sum.values)
seed_arr = reg.is_seed.values

share = seed_arr.astype(float)
for it in range(300):
    contrib = np.bincount(d, weights=w * share[s], minlength=len(reg))
    new = np.where(denom > 0, np.minimum(contrib / np.where(denom > 0, denom, 1), 1), 0)
    new[seed_arr] = 1.0
    if np.abs(new - share).max() < 1e-10:
        share = new; break
    share = new
contrib = np.bincount(d, weights=w * share[s], minlength=len(reg))
reg["seed_share"] = share             # какая доля денег узла — предположительно от seed
reg["seed_money_in"] = contrib        # сколько «seed-денег» пришло, KZT
print("Итераций до сходимости:", it + 1)

top_seed = set(reg.nlargest(100, "seed_money_in").index)
low = reg.loc[list(top_tv)].query("seed_share < 0.1 and not is_seed")
note(4, "пересечение топ-100 по обороту и по seed-деньгам", len(top_tv & top_seed),
     "оборот плохо заменяет связь с seed" if len(top_tv & top_seed) < 70 else "метрики близки")
note(4, "узлов в топ-100 по обороту, где seed-денег < 10%", len(low),
     "кандидаты в «посторонние» крупные счета")
note(4, "Spearman: оборот vs seed-деньги",
     round(reg[["total_volume", "seed_money_in"]].corr("spearman").iloc[0, 1], 2))

print("\nДоля seed-денег по коленам:")
display(reg.groupby("depth").seed_share.describe()[["mean", "25%", "50%", "75%"]])
display(low.sort_values("total_volume", ascending=False).head(10)
        [["depth", "total_volume", "in_sum", "seed_money_in", "seed_share", "in_deg"]])

Итераций до сходимости: 121
[4] пересечение топ-100 по обороту и по seed-деньгам: 20
     → оборот плохо заменяет связь с seed
[4] узлов в топ-100 по обороту, где seed-денег < 10%: 77
     → кандидаты в «посторонние» крупные счета
[4] Spearman: оборот vs seed-деньги: 0.39

Доля seed-денег по коленам:


,mean,25%,50%,75%
depth,,,,
0,1,1,1,1
1,0.7697,0.5349,1,1
2,0.2099,0.01719,0.07957,0.2493
3,0.02293,0.0006645,0.001845,0.007631
4,0.008658,0.0002427,0.001137,0.004209


,depth,total_volume,in_sum,seed_money_in,seed_share,in_deg
gid,,,,,,
100000000331309100,2,"23,986,010","984,635","27,495",0.001195,5
100000002224132100,2,"16,066,020","3,951,020","181,059",0.01495,5
100000005242320100,2,"9,301,995","365,700","2,866",0.0003207,2
100000000437046100,2,"8,713,424","1,107,200","36,890",0.00485,9
100000008603629100,2,"6,803,456","1,817,300","33,165",0.006652,19
100000004400305100,2,"6,052,981","222,800","4,935",0.0008464,7
100000002963189100,3,"5,709,000","490,000",542,0.0001038,1
100000008710791100,2,"5,703,021","181,510","6,961",0.001261,3
100000008547948100,2,"5,398,265","212,665","5,830",0.001124,6


In [18]:
hubs = reg[reg.in_deg >= 8].copy()
hubs["avg_payment"]      = hubs.in_sum / hubs.in_tx
hubs["in_days"]          = tx.groupby("dst").date.apply(lambda x: x.dt.date.nunique())
hubs["amount_cv"]        = tx.groupby("dst").sum_kzt.agg(lambda x: x.std() / x.mean() if len(x) > 1 else np.nan)
payer_outdeg = e.src.map(reg.out_deg)
hubs["payers_exclusive"] = e.assign(p=payer_outdeg == 1).groupby("dst").p.mean()   # доля плательщиков, платящих только ему

merchant_like  = (hubs.seed_share < 0.2) & (hubs.in_days >= 15) & (hubs.pass_ratio.fillna(0) < 0.2)
collector_like = (hubs.seed_share >= 0.5) & (hubs.payers_exclusive >= 0.5)

note(5, "узлов с 8+ плательщиками", len(hubs))
note(5, "похожи на легальный хаб (мало seed-денег, вход ≥15 дней, почти не отдают)", int(merchant_like.sum()),
     "не называть их консолидаторами без оговорки")
note(5, "похожи на сборщика (≥50% seed-денег, плательщики эксклюзивны)", int(collector_like.sum()))

cols5 = ["depth", "in_deg", "in_sum", "seed_share", "in_days", "avg_payment", "amount_cv",
         "payers_exclusive", "pass_ratio", "out_deg"]
print("\nПохожи на хаб:");     display(hubs[merchant_like].sort_values("in_deg", ascending=False)[cols5])
print("\nПохожи на сборщика:"); display(hubs[collector_like].sort_values("in_deg", ascending=False)[cols5])

[5] узлов с 8+ плательщиками: 17
[5] похожи на легальный хаб (мало seed-денег, вход ≥15 дней, почти не отдают): 0
     → не называть их консолидаторами без оговорки
[5] похожи на сборщика (≥50% seed-денег, плательщики эксклюзивны): 0

Похожи на хаб:


,depth,in_deg,in_sum,seed_share,in_days,avg_payment,amount_cv,payers_exclusive,pass_ratio,out_deg
gid,,,,,,,,,,



Похожи на сборщика:


,depth,in_deg,in_sum,seed_share,in_days,avg_payment,amount_cv,payers_exclusive,pass_ratio,out_deg
gid,,,,,,,,,,


In [19]:
reg["pagerank"]    = pd.Series(nx.pagerank(G, weight="sum_kzt"))
reg["betweenness"] = pd.Series(nx.betweenness_centrality(G))   # точный расчёт, несколько секунд

for m in ["pagerank", "betweenness"]:
    t = reg.nlargest(50, m)
    note(6, f"топ-50 по {m}: распределение по коленам",
         t.depth.value_counts().sort_index().to_dict(),
         f"обрезанных узлов: {int(t.flag_truncated.sum())}")

print("\nСредние по коленам:")
display(reg.groupby("depth")[["pagerank", "betweenness"]].mean())

[6] топ-50 по pagerank: распределение по коленам: {0: 7, 1: 12, 2: 21, 3: 10}
     → обрезанных узлов: 0
[6] топ-50 по betweenness: распределение по коленам: {0: 4, 1: 15, 2: 20, 3: 11}
     → обрезанных узлов: 0

Средние по коленам:


,pagerank,betweenness
depth,,
0,0.0005991,0.0002172
1,0.0004266,8.183e-05
2,0.0005506,0.0001372
3,0.0003907,4.089e-05
4,0.0004222,0


In [20]:
sd = reg[reg.is_seed]
in_edges = set(e.src) | set(e.dst)
only_recv = sd[(sd.out_deg == 0) & (sd.in_deg > 0)]

note(7, "seed без исходящих", int((sd.out_deg == 0).sum()), "в ТЗ 31")
note(7, "seed вообще нет в рёбрах", int((~sd.index.isin(list(in_edges))).sum()), "в ТЗ 19; роль = «нет данных»")
note(7, "seed только получатели", len(only_recv), "в ТЗ 12")
note(7, "seed, получающие деньги от других seed", int((sd.in_from_seed_deg > 0).sum()),
     "связи между seed — сигнал общей структуры")

# кто платит seed-получателям: другие seed или узлы глубже (возвратный поток)
payers = edge_reg[edge_reg.dst.isin(only_recv.index)]
print("\nКолено плательщиков для seed-получателей:")
display(payers.src_depth.value_counts().sort_index())
print("\nSeed по компонентам:")
display(sd.groupby("component").size().rename("n_seed").to_frame().join(
        reg.groupby("component").size().rename("component_size")))

[7] seed без исходящих: 31
     → в ТЗ 31
[7] seed вообще нет в рёбрах: 19
     → в ТЗ 19; роль = «нет данных»
[7] seed только получатели: 12
     → в ТЗ 12
[7] seed, получающие деньги от других seed: 18
     → связи между seed — сигнал общей структуры

Колено плательщиков для seed-получателей:


src_depth
0    7
1    5
2    9
3    1
Name: count, dtype: int64


Seed по компонентам:


,n_seed,component_size
component,,
0,46,1877
1,1,270
2,1,17
3,1,13
4,1,6
5,1,6
6,1,6
7,1,6
8,2,5


In [21]:
amt = tx.sum_kzt
note(8, "минимальная сумма транзакции", fmt(amt.min()), "порог соблюдён" if amt.min() >= 5000 else "есть суммы ниже порога!")
note(8, "доля транзакций 5–6 тыс.", f"{amt.between(5000, 5999.99).mean():.1%}",
     "скопление у порога = признак, что ниже есть ещё")
note(8, "доля круглых сумм (кратно 1000)", f"{(amt % 1000 == 0).mean():.1%}")

same_day = tx.groupby(["src", "dst", tx.date.dt.date]).size()
note(8, "пар «отправитель–получатель–день» с 2+ переводами", int((same_day >= 2).sum()),
     "кандидаты в дробление выше порога")

bins = [5000, 6000, 7000, 8000, 9000, 10000, 15000, 20000, 50000, 100000, 500000, np.inf]
display(pd.cut(amt, bins, right=False).value_counts().sort_index().rename("n_tx").to_frame())
display(same_day[same_day >= 2].sort_values(ascending=False).head(15).rename("n_tx_same_day").to_frame())

[8] минимальная сумма транзакции: 5 000
     → порог соблюдён
[8] доля транзакций 5–6 тыс.: 6.9%
     → скопление у порога = признак, что ниже есть ещё
[8] доля круглых сумм (кратно 1000): 65.6%
[8] пар «отправитель–получатель–день» с 2+ переводами: 375
     → кандидаты в дробление выше порога


,n_tx
sum_kzt,
"[5000.0, 6000.0)",335
"[6000.0, 7000.0)",135
"[7000.0, 8000.0)",103
"[8000.0, 9000.0)",88
"[9000.0, 10000.0)",95
"[10000.0, 15000.0)",588
"[15000.0, 20000.0)",366
"[20000.0, 50000.0)",1369
"[50000.0, 100000.0)",705


n_tx_same_day
src                dst                date                     
100000004269433100 100000008418835100 2026-07-12             11
                                      2026-07-16              9
                                      2026-07-14              9
                                      2026-07-09              8
100000000437046100 100000007055802100 2026-07-15              8
100000008628231100 100000005664632100 2026-07-26              8
100000007390016100 100000008324800100 2026-07-07              7
100000008324800100 100000007390016100 2026-07-17              7
100000007170072100 100000003345947100 2026-07-13              6
100000008637537100 100000001732159100 2026-07-04              6
100000004269433100 100000008418835100 2026-07-18              6
100000004388983100 100000005034574100 2026-07-28              6
                                      2026-07-31              6
100000002645993100 100000002398779100 2026-07-07              5
100000008637537100 100000001732159100 2026-07-13              5

In [22]:
note(9, "период транзакций", f"{tx.date.min().date()} — {tx.date.max().date()}")
last_in = tx.groupby("dst").date.max().reindex(reg.index[real_sink])
late = last_in >= tx.date.max() - pd.Timedelta(days=3)
note(9, "настоящих стоков с последним входом в последние 3 дня", int(late.sum()),
     "могли переслать деньги уже в августе, terminal под сомнением")

daily = tx.groupby(tx.date.dt.date).sum_kzt.agg(n_tx="size", sum_kzt="sum")
display(daily)

[9] период транзакций: 2026-07-01 — 2026-07-31
[9] настоящих стоков с последним входом в последние 3 дня: 171
     → могли переслать деньги уже в августе, terminal под сомнением


,n_tx,sum_kzt
date,,
2026-07-01,115,"7,179,461"
2026-07-02,131,"9,362,006"
2026-07-03,181,"12,619,868"
2026-07-04,110,"5,949,727"
2026-07-05,127,"7,709,834"
2026-07-06,124,"5,313,603"
2026-07-07,150,"9,882,395"
2026-07-08,190,"16,175,687"
2026-07-09,156,"10,483,391"


In [23]:
note(10, "edges.depth == колено отправителя", f"{(edge_reg.depth == edge_reg.src_depth).mean():.0%}")
note(10, "edges.depth == колено отправителя + 1", f"{(edge_reg.depth == edge_reg.src_depth + 1).mean():.0%}")
note(10, "edges.depth == колено получателя", f"{(edge_reg.depth == edge_reg.dst_depth).mean():.0%}")
display(pd.crosstab(edge_reg.depth, edge_reg.src_depth, margins=True))

back = edge_reg[edge_reg.back_edge]
note(10, "обратных/боковых рёбер", f"{len(back)} на {fmt(back.sum_kzt.sum())} KZT")
note(10, "из них ведут в seed", int(back.dst_is_seed.sum()), "деньги возвращаются к известным участникам")
note(10, "взаимных пар A↔B", int(edge_reg.mutual.sum() // 2))

from itertools import islice
try:
    cycles = list(nx.simple_cycles(G, length_bound=5))
except TypeError:                      # старый networkx без length_bound
    cycles = [c for c in islice(nx.simple_cycles(G), 20000) if len(c) <= 5]
reg["in_cycle"] = reg.index.isin({n for c in cycles for n in c})
note(10, "циклов длиной ≤5", len(cycles), f"узлов в циклах: {int(reg.in_cycle.sum())}")
display(pd.Series([len(c) for c in cycles]).value_counts().sort_index().rename("n_cycles").to_frame())

[10] edges.depth == колено отправителя: 0%
[10] edges.depth == колено отправителя + 1: 100%
[10] edges.depth == колено получателя: 76%


src_depth,0,1,2,3,All
depth,,,,,
1,520,0,0,0,520
2,0,640,0,0,640
3,0,0,1200,0,1200
4,0,0,0,759,759
All,520,640,1200,759,3119


[10] обратных/боковых рёбер: 745 на 84 279 596 KZT
[10] из них ведут в seed: 125
     → деньги возвращаются к известным участникам
[10] взаимных пар A↔B: 177
[10] циклов длиной ≤5: 468
     → узлов в циклах: 300


,n_cycles
2,177
3,41
4,169
5,81


In [24]:
from sklearn.metrics import adjusted_rand_score

# неориентированный граф с суммой весов в обе стороны
U = nx.Graph(); U.add_nodes_from(G)
for u, v, dd in G.edges(data=True):
    if U.has_edge(u, v): U[u][v]["w"] += dd["sum_kzt"]
    else: U.add_edge(u, v, w=dd["sum_kzt"])
for _, _, dd in U.edges(data=True):
    dd["logw"] = np.log1p(dd["w"])

def labels(comms):
    lab = {n: i for i, c in enumerate(comms) for n in c}
    return np.array([lab[n] for n in reg.index])

runs = {(wt, sd_): labels(nx.community.louvain_communities(U, weight=wt, seed=sd_))
        for wt in ["logw", "w", None] for sd_ in range(5)}

for wt in ["logw", "w", None]:
    aris = [adjusted_rand_score(runs[(wt, 0)], runs[(wt, k)]) for k in range(1, 5)]
    note(11, f"устойчивость Louvain между запусками (вес={wt}), ARI", round(np.mean(aris), 3),
         "стабильно" if np.mean(aris) > 0.9 else "кластеры плавают, фиксировать seed и проверять")
note(11, "совпадение разбиений log-вес vs сырой вес, ARI",
     round(adjusted_rand_score(runs[("logw", 0)], runs[("w", 0)]), 3), "выбор веса влияет на кластеры")

lab0 = runs[("logw", 0)]
cs = (pd.DataFrame({"c": lab0, "seed": reg.is_seed.values, "comp": reg.component.values})
        .groupby("c").agg(n=("seed", "size"), seeds=("seed", "sum"), n_comp=("comp", "nunique")))
note(11, "сообществ всего / ≥5 узлов / с >1 seed",
     f"{len(cs)} / {(cs.n >= 5).sum()} / {(cs.seeds > 1).sum()}", "в ТЗ ориентир 8 сообществ с >1 seed")
note(11, "сообществ внутри крупнейшей компоненты", int(cs[cs.index.isin(np.unique(lab0[reg.component.values == 0]))].shape[0]))
reg["louvain"] = lab0
display(cs.sort_values("n", ascending=False).head(15))

[11] устойчивость Louvain между запусками (вес=logw), ARI: 0.675
     → кластеры плавают, фиксировать seed и проверять
[11] устойчивость Louvain между запусками (вес=w), ARI: 0.899
     → кластеры плавают, фиксировать seed и проверять
[11] устойчивость Louvain между запусками (вес=None), ARI: 0.727
     → кластеры плавают, фиксировать seed и проверять
[11] совпадение разбиений log-вес vs сырой вес, ARI: 0.432
     → выбор веса влияет на кластеры
[11] сообществ всего / ≥5 узлов / с >1 seed: 64 / 39 / 9
     → в ТЗ ориентир 8 сообществ с >1 seed
[11] сообществ внутри крупнейшей компоненты: 28


,n,seeds,n_comp
c,,,
24,203,1,1
3,174,13,1
34,169,1,1
8,135,4,1
27,120,0,1
1,111,1,1
10,108,0,1
0,103,1,1
41,91,0,1


In [25]:
metrics = ["total_volume", "volume_dedup", "in_sum", "out_sum", "throughput", "seed_money_in",
           "in_deg", "out_deg", "n_counterparties", "pagerank", "betweenness"]
tops = {m: set(reg.nlargest(100, m).index) for m in metrics}

print("Пересечения топ-100 (из 100):")
display(pd.DataFrame([[len(tops[a] & tops[b]) for b in metrics] for a in metrics], index=metrics, columns=metrics))
print("Ранговые корреляции Spearman:")
display(reg[metrics].corr("spearman").round(2))

cnt = pd.Series([g for m in metrics for g in tops[m]]).value_counts()
reg["n_top_lists"] = cnt.reindex(reg.index).fillna(0).astype(int)
note(12, "узлов в топ-100 хотя бы по одной метрике", len(cnt))
note(12, "узлов в топ-100 по ≥6 метрикам из 11", int((cnt >= 6).sum()),
     "устойчивое ядро, кандидаты в начало приоритетного списка")
display(reg.loc[cnt[cnt >= 6].index,
        ["depth", "is_seed", "n_top_lists", "total_volume", "seed_money_in", "in_deg", "out_deg",
         "pass_ratio", "flag_truncated"]].sort_values("n_top_lists", ascending=False))

Пересечения топ-100 (из 100):


,total_volume,volume_dedup,in_sum,out_sum,throughput,seed_money_in,in_deg,out_deg,n_counterparties,pagerank,betweenness
total_volume,100,91,44,72,54,20,32,49,47,19,38
volume_dedup,91,100,40,73,45,20,29,52,49,16,39
in_sum,44,40,100,16,36,24,31,11,14,26,12
out_sum,72,73,16,100,48,13,25,62,59,10,45
throughput,54,45,36,48,100,29,36,31,33,27,25
seed_money_in,20,20,24,13,29,100,20,8,10,23,7
in_deg,32,29,31,25,36,20,100,29,41,27,34
out_deg,49,52,11,62,31,8,29,100,88,3,64
n_counterparties,47,49,14,59,33,10,41,88,100,7,61
pagerank,19,16,26,10,27,23,27,3,7,100,7


Ранговые корреляции Spearman:


,total_volume,volume_dedup,in_sum,out_sum,throughput,seed_money_in,in_deg,out_deg,n_counterparties,pagerank,betweenness
total_volume,1,0.99,0.86,0.55,0.53,0.39,0.47,0.52,0.6,0.42,0.49
volume_dedup,0.99,1,0.87,0.48,0.46,0.38,0.44,0.45,0.55,0.41,0.42
in_sum,0.86,0.87,1,0.19,0.28,0.43,0.48,0.18,0.34,0.46,0.21
out_sum,0.55,0.48,0.19,1,0.96,0.24,0.36,0.99,0.81,0.22,0.9
throughput,0.53,0.46,0.28,0.96,1,0.31,0.44,0.95,0.81,0.28,0.91
seed_money_in,0.39,0.38,0.43,0.24,0.31,1,0.38,0.25,0.32,0.39,0.25
in_deg,0.47,0.44,0.48,0.36,0.44,0.38,1,0.37,0.7,0.39,0.44
out_deg,0.52,0.45,0.18,0.99,0.95,0.25,0.37,1,0.83,0.21,0.91
n_counterparties,0.6,0.55,0.34,0.81,0.81,0.32,0.7,0.83,1,0.26,0.84
pagerank,0.42,0.41,0.46,0.22,0.28,0.39,0.39,0.21,0.26,1,0.19


[12] узлов в топ-100 хотя бы по одной метрике: 419
[12] узлов в топ-100 по ≥6 метрикам из 11: 56
     → устойчивое ядро, кандидаты в начало приоритетного списка


,depth,is_seed,n_top_lists,total_volume,seed_money_in,in_deg,out_deg,pass_ratio,flag_truncated
100000003684369100,0,True,11,"12,437,091","500,392",24,62,2.232,False
100000008603629100,2,False,10,"6,803,456","33,165",19,61,2.744,False
100000008686313100,1,False,9,"2,246,516","233,150",6,7,0.1745,False
100000006866783100,0,True,9,"4,786,849","38,422",13,67,4.707,False
100000003016635100,0,True,9,"10,001,062","106,066",8,73,16.04,False
100000001530983100,1,False,9,"2,393,696","393,443",4,11,3.243,False
100000005933757100,2,False,9,"4,248,628","3,320",4,34,2.41,False
100000000437046100,2,False,9,"8,713,424","36,890",9,42,6.87,False
100000001857829100,2,False,9,"3,279,528","3,996",6,16,2.691,False
100000008165763100,1,False,9,"2,433,673","188,882",15,17,1.088,False


In [26]:
res = pd.DataFrame(findings)
display(res)
res.to_csv("checks_findings.csv", index=False, encoding="utf-8-sig")
reg.reset_index().to_csv("registry_nodes_enriched.csv", index=False, encoding="utf-8-sig")

,пункт,проверка,значение,вывод
0,1,доля транзитных (pass 0.8–1.2): топ-100 по in+...,2% vs 3%,сильного перекоса нет
1,1,"пересечение топ-100 по in+out и по max(in,out)",91,почти не меняется
2,2,не-seed узлов с out > in,335,"у них не виден часть входа, баланс считать нельзя"
3,2,"невидимый вход у них в сумме, KZT",229 908 004,= 62.8% оборота графа
4,2,из них первый исходящий раньше первого видимог...,186,"деньги были на счёте до выборки, т.е. вход из-..."
5,3,узлов без исходящих всего,1554,
6,3,из них обрыв 4-го колена,444,"не terminal, а «нет данных»"
7,3,"кандидаты в настоящие стоки (колено<4, не seed...",1079,сумма входа 136 739 480 KZT
8,3,"доля денег графа, ушедших в обрезанные узлы",15.5%,
9,3,обрезанных в топ-100 по входу,17,наивное правило «много пришло и не ушло» пойма...
